# Thesis: Reclaimed Timber in Deep Generative Design
**Notebook ID:** 11_12_15_main_geometry_generation  
**Author:** Jasper Cluistra   
**Last Updated:** 2026-03-03
### Geometry generation script
**Goal:** Generate the geometry dataset used as input for the Grasshopper structural analysis. The script accepts several variables and writes a CSV file containing the properties needed to reconstruct the digital geometry as CAD geometry.

**Inputs:**   
*   Base mesh definition
*   Allowed movement for different vertices
*   Divisions over the allowed movement

**Outputs:**

*   CSV file with sample ID, vertices, normalized coordinates, and edges
*   Centroid and PCA alignment are applied once in generation; do not repeat that normalization in Grasshopper

### Testing corners

In [ ]:
import c00_headquarter_params as c11_params
import config
from c12_geometry_truss import get_corner_indices

# ── Test ───────────────────────────────────────────────────────────────────────
grid_2x2 = get_corner_indices(2, 2)
print(f"2x2 grid indices: {grid_2x2}")

grid_3x3 = get_corner_indices(3, 3)
print(f"3x3 grid indices: {grid_3x3}")

grid_5x3 = get_corner_indices(5, 3)
print(f"5x3 grid indices: {grid_5x3}")
# Expected: 0 (BL), 4 (BR), 20 (TL), 24 (TR) -> because there are 5 points per row

# ── In the loop ───────────────────────────────────────────────────────────────────────
corners = get_corner_indices(c11_params.GRID_CELLS_X, c11_params.GRID_CELLS_Y)
corner_values = corners.values() # [0, 4, 20, 24] for example
print(corner_values)

## 1.2 Executing and saving dataset

In [ ]:
import pandas as pd
import numpy as np
import config
import c00_headquarter_params as c11_params
from c12_geometry_truss import generate_vertices, generate_edges

# ── Run ───────────────────────────────────────────────────────────────────────
final_vertices = generate_vertices(c11_params.NUM_SAMPLES)
final_edges = generate_edges(c11_params.NUM_SAMPLES, c11_params.GRID_CELLS_X, c11_params.GRID_CELLS_Y)

# ── Save ───────────────────────────────────────────────────────────────────────
final_vertices_name = f"dataset_{c11_params.GRID}_{c11_params.NUM_SAMPLES}_vertices.csv"
final_vertices.to_csv(config.GEOM_DATA_PATH / final_vertices_name, index=False)
final_edges.to_csv(config.GEOM_DATA_PATH / f"dataset_{c11_params.GRID}_{c11_params.NUM_SAMPLES}_edges.csv", index=False)

print(f"Generation complete for {c11_params.NUM_SAMPLES} samples.")
print(f"Grid size: {c11_params.GRID}")

print("Vertices shape:", final_vertices.shape)
print("Edges shape:", final_edges.shape)

print(f"\nTotal rows in vertices CSV: {len(final_vertices)}")
print("\nExample output (first 5 rows of sample 0):")
print(final_vertices.head(5))
print("\nExample output (first 5 rows of sample 1):")
print(final_vertices[final_vertices['sample_id'] == 1].head(5))

print(f"\nTotal rows in edges CSV: {len(final_edges)}")
print("\nExample output (first 5 rows of sample 0):")
print(final_edges.head(5))
print("\nExample output (first 5 rows of sample 1):")
print(final_edges[final_edges['sample_id'] == 1].head(5))

## 1.5 DESIGN DOMAIN
two output from the geometry script need to be transferred to json files for further use in the script, these are:
*   Search Space, this is the space where the vertices can move, this is nessecary for the optimizing of the geometry so the optimizer knows where it can and cant go.
*   Edge Index, the edge index is nessecary for the training of the surrogate model, this is used because the relationship of the vertices is the same vor every structure, also lowers this the number of features (x) used in training for a better generalisation.

In [ ]:
import json
import config
import c00_headquarter_params as c11_params
from c12_geometry_truss import generate_edges, define_search_space

# ── Run and Review ───────────────────────────────────────────────────────────────
# Use the variables from your earlier configuration
search_space = define_search_space(c11_params.GRID_CELLS_X, c11_params.GRID_CELLS_Y, c11_params.DIVISIONS, c11_params.EDGE_LENGTH)
df_topology = generate_edges(num_samples=1, cells_x=c11_params.GRID_CELLS_X, cells_y=c11_params.GRID_CELLS_Y)
edge_index = df_topology[['V1', 'V2']].values.T.tolist()

print(f"The search space contains {len(search_space)} independent variables (the controls the AI can adjust).")
print(f"Topology generated. Number of unique connections (edges): {len(df_topology)}")
print("\nExample of how the computer sees this:")

# Show the first 5 variables in a readable format
for var_name, bounds in list(search_space.items())[:5]:
    print(f"- {var_name}: {bounds}")

print("\nGenerated edge_index (first 5 connections):")
print(f"Source nodes (V1): {edge_index[0][:5]}")
print(f"Target nodes (V2): {edge_index[1][:5]}")

# Save the dictionaries as JSON files
json_search_space_path = config.DATA_IO_PATH / f"search_space_{c11_params.GRID}.json"
json_edge_index_path = config.DATA_IO_PATH / f"edge_index_{c11_params.GRID}.json"
with open(json_search_space_path, 'w') as f:
    json.dump(search_space, f, indent=4) # indent=4 makes it easy to read
with open(json_edge_index_path, 'w') as f:
    json.dump(edge_index, f)

print(f"Search space successfully saved as '{json_search_space_path}'")
print(f"\nEdge index (as a plain list) successfully saved to '{json_edge_index_path}'.")

## Average Length
Here the average length of the geometry is calculated for the generation of the timber stock

In [ ]:
import config
import json
import importlib
import pandas as pd
import c00_headquarter_params as c11_params
import c12_reconstruction

importlib.reload(c12_reconstruction)

# Load vertices and edge index from disk so this cell can run independently
final_vertices_path = config.GEOM_DATA_PATH / f"dataset_{c11_params.GRID}_{c11_params.NUM_SAMPLES}_vertices.csv"
with open(config.DATA_IO_PATH / "edge_index.json", 'r', encoding='utf-8') as f:
    edge_index = json.load(f)

final_vertices = pd.read_csv(final_vertices_path)

# Generate material statistics using the loaded vertices dataframe
result = c12_reconstruction.generate_material_statistics(
    df_vertices=final_vertices,
    edge_index=edge_index,
    sample_count=10,
    random_state=42
)

summary = result["summary_statistics"]

print("Representative statistics:")
print(f"  Representative length: {result['representative_length_m']:.3f}m ({result['representative_length_mm']:.1f}mm)")
print(f"  Average length: {summary['average_length_m']:.3f}m ({summary['average_length_mm']:.1f}mm)")
print(f"  Median length: {summary['median_length_m']:.3f}m ({summary['median_length_mm']:.1f}mm)")
print(f"  Min length: {summary['min_length_m']:.3f}m ({summary['min_length_mm']:.1f}mm)")
print(f"  Max length: {summary['max_length_m']:.3f}m ({summary['max_length_mm']:.1f}mm)")
print(f"  Total length: {summary['total_length_m']:.3f}m ({summary['total_length_mm']:.1f}mm)")
print(f"  Std dev: {summary['std_dev_m']:.3f}m ({summary['std_dev_mm']:.1f}mm)")
print(f"  Q1: {summary['q1_m']:.3f}m ({summary['q1_mm']:.1f}mm)")
print(f"  Q3: {summary['q3_m']:.3f}m ({summary['q3_mm']:.1f}mm)")
print(f"  Average edge count: {summary['edge_count']}")
print(f"Selected samples: {result['selected_sample_ids']}\n")

print("Per-sample statistics:")
for sample_result in result['per_sample_results']:
    print(f"\nSample {sample_result['sample_id']}:")
    print(f"  Average: {sample_result['average_length_m']:.3f}m ({sample_result['average_length_mm']:.1f}mm)")
    print(f"  Median: {sample_result['median_length_m']:.3f}m ({sample_result['median_length_mm']:.1f}mm)")
    print(f"  Min: {sample_result['min_length_m']:.3f}m ({sample_result['min_length_mm']:.1f}mm)")
    print(f"  Max: {sample_result['max_length_m']:.3f}m ({sample_result['max_length_mm']:.1f}mm)")
    print(f"  Total: {sample_result['total_length_m']:.3f}m ({sample_result['total_length_mm']:.1f}mm, {sample_result['edge_count']} beams)")

# Save to JSON
with open(config.DATA_IO_PATH / "representative_beam_statistics.json", 'w', encoding='utf-8') as f:
    json.dump(result, f, indent=2)

## 2.0 GNN Retraining — Training Data Generation (Phase 1)

Generates node and edge feature CSVs for the GNN surrogate model retraining pipeline.

**Strategy:** random stock assignment across randomly sampled geometries — no MILP or cost-matrix pipeline. The GNN learns structural physics (geometry + cross-section → utilization), not cost-optimality. Random assignment gives full coverage of safe and unsafe combinations across the whole stock pool.

**Mean-EA force estimation** (c24) is retained as a fast single FEM solve per geometry to produce `N_mean_EA` — the structural demand per member, independent of stock assignment.

**Outputs** (saved to `config.GH_DATA_PATH`):
- `training_nodes_raw.csv` — node positions + boundary conditions + layer/attribute, one row per node per sample
- `training_edges_raw.csv` — edge features **without** Utilization (Karamba FEA adds this column)

**Next step:** open Grasshopper, load both CSVs, run Karamba FEA, and export an updated edges CSV with a `Utilization` column. Then retrain with `c21_surrogate_training.run_preprocessing()`.

In [1]:
import importlib
import config
from workflows import c12_generate_training_data as gen
importlib.reload(gen)

# =============================================================================
# PARAMETERS — adjust these before running
# =============================================================================

# Total number of training samples to generate.
# Each sample: random geometry + mean-EA FEM solve + random stock assignment.
# ~0.3–0.5 s per sample locally → ~1–1.5 h for 10 000 samples.
N_SAMPLES = 20_000

# Random seed — keep fixed so the dataset is reproducible.
RANDOM_SEED = 42

# Print per-sample details (True = every sample, False = every 50 samples).
VERBOSE = False

# Save checkpoint CSVs every N samples for crash recovery (0 = disabled).
SAVE_INTERMEDIATE = 0

# Output directory — defaults to config.GH_DATA_PATH when None.
OUTPUT_DIR = None   # e.g. config.GH_DATA_PATH / "training_v2"

# =============================================================================
# RUN
# =============================================================================

result = gen.generate_training_samples(
    n_samples          = N_SAMPLES,
    random_seed        = RANDOM_SEED,
    verbose            = VERBOSE,
    save_intermediate  = SAVE_INTERMEDIATE,
    output_dir         = OUTPUT_DIR,
)

Config System loaded successfully, Code running locally from thesis_generative_timber and Data is connected to OneDrive 2.2 - 2.4.

GRID: 5x3, EDGE_LENGTH: 3.0, LAYER_HEIGHT: 1.5, DIVISIONS: 8, NUM_SAMPLES: 20000

IMPACT_FACTOR_A1_A3: 0.25, IMPACT_FACTOR_RECOVERED_C1: 0.0085, "ENERGY_PREP_SAW_A5: 0.02, ENERGY_OFFCUT_FACTOR_C3_C4: 0.276, SCARCITY_PENALTY: 0

parameters loaded from c:\Users\Jasper\Documents\PyRepo\thesis_generative_timber\c00_headquarter_params.py
[c12_generate_training_data] 20260516_001012
  Stock:         complete_timber_v2.csv
  Search space:  search_space.json
  Output dir:    C:\Users\Jasper\OneDrive\06 Building Technology TU\2.2 - 2.4\30_Data_Inventory\02_grasshopper_data
  Target:        20000 samples  |  seed: 42

  Stock loaded: 526 elements
  Topology:      120 edges, 5x3 grid

--- Generating 20000 samples ---
  [00:10:15]     100/20000  (attempts=100, skipped=0)
  [00:10:18]     200/20000  (attempts=200, skipped=0)
  [00:10:21]     300/20000  (attempts=300, s

In [3]:
# --- INSPECT OUTPUT ---
import pandas as pd

df_nodes = result["df_nodes"]
df_edges = result["df_edges_out"]

print(f"Samples generated:  {result['n_generated']}  (skipped: {result['n_skipped']})")
print(f"\nNode CSV:  {result['nodes_csv_path'].name}  ({len(df_nodes)} rows)")
print(f"Edge CSV:  {result['edges_csv_path'].name}  ({len(df_edges)} rows)")

print(f"\nEdge feature columns (GNN input after Karamba adds Utilization):")
print(f"  {gen.NEW_EDGE_COLS}")

print(f"\nN_mean_EA range:  "
      f"{df_edges['N_mean_EA'].min():.0f} N  to  {df_edges['N_mean_EA'].max():.0f} N")
print(f"Width_m range:    "
      f"{df_edges['Width_m'].min()*1000:.0f} mm  to  {df_edges['Width_m'].max()*1000:.0f} mm")
print(f"Depth_m range:    "
      f"{df_edges['Depth_m'].min()*1000:.0f} mm  to  {df_edges['Depth_m'].max()*1000:.0f} mm")

print(f"\nFirst 3 edge rows (sample 0):")
display(df_edges[df_edges["sample_id"] == 0][gen.NEW_EDGE_COLS + ["edge_id"]].head(3))

print(f"\nFirst 3 node rows (sample 0):")
display(df_nodes[df_nodes["sample_id"] == 0].head(3))

Samples generated:  20000  (skipped: 0)

Node CSV:  training_nodes_raw.csv  (780000 rows)
Edge CSV:  training_edges_raw.csv  (2400000 rows)

Edge feature columns (GNN input after Karamba adds Utilization):
  ['Width_m', 'Depth_m', 'Length', 'E', 'Iy', 'Iz', 'J', 'EA/L', 'N_mean_EA']

N_mean_EA range:  -76479 N  to  116240 N
Width_m range:    38 mm  to  300 mm
Depth_m range:    100 mm  to  300 mm

First 3 edge rows (sample 0):


,Width_m,Depth_m,Length,E,Iy,Iz,J,EA/L,N_mean_EA,edge_id
0,0.063,0.175,4.125,1.100000e+10,0.000028,0.000004,0.000011,29400000.0,-2587.27,e0
1,0.100,0.100,3.750,1.100000e+10,0.000008,0.000008,0.000012,29333333.3,-1703.64,e1
2,0.075,0.225,2.625,1.100000e+10,0.000071,0.000008,0.000025,70714285.7,5963.87,e2



First 3 node rows (sample 0):


,sample_id,vertex_index,layer,attribute,x,y,z,Tx,Ty,Tz,Rx,Ry,Rz,Fz
0,0,v0,top,support,-7.608,-4.318,0.567,1.0,1.0,1.0,0.0,0.0,0.0,0.0
1,0,v1,top,load,-3.483,-4.318,0.567,0.0,0.0,0.0,0.0,0.0,0.0,-13500.0
2,0,v2,top,load,-0.858,-4.318,0.567,0.0,0.0,0.0,0.0,0.0,0.0,-13500.0
